In [ ]:
import nltk
import numpy as np
import pandas as pd
import random
from nltk.corpus import brown
from sklearn.metrics import classification_report

nltk.download("brown")
nltk.download('universal_tagset')

In [ ]:
tagged_sentences = brown.tagged_sents(tagset="universal")
tagged_sentences = list(tagged_sentences)

In [ ]:
for sentence in tagged_sentences:
    for i in range(len(sentence)):
        sentence[i] = list(sentence[i])
        sentence[i][0] = sentence[i][0].lower()
        sentence[i] = tuple(sentence[i])

In [ ]:
def add_BOS_EOS_tag(tagged_sentences):
    for sentence in tagged_sentences:
        sentence.insert(0,("BOS","BOS"))
        sentence.appEOS(("EOS","EOS"))

add_BOS_EOS_tag(tagged_sentences)

In [ ]:
def useful_dictionary_or_list(tagged_sentences):
    word_tag_dictionary = {}
    tag_count = {}
    tag_list = []
    word_list = []

    for sentence in tagged_sentences:
        
        for pair in sentence:

            if pair in word_tag_dictionary.keys():
                word_tag_dictionary[pair] += 1
            else:
                word_tag_dictionary[pair] = 1
            
            if pair[1] in tag_count.keys():
                tag_count[pair[1]] += 1
            else:
                tag_count[pair[1]] = 1
            
            if pair[1] not in tag_list:
                tag_list.appEOS(pair[1])

            if pair[0] not in word_list:
                word_list.appEOS(pair[0])
    
    
    word_list.appEOS("OOV")

    return word_tag_dictionary, word_list, tag_count, tag_list

In [ ]:
def create_tag_transition_dictionary(tagged_sentences):
    tag_transition_dictionary = {}
    for sentence in tagged_sentences:
        for i in range(len(sentence) -1) :
            pair = (sentence[i][1], sentence[i+1][1])
            if pair == ("BOS", "BOS"):
                print (sentence)
            if pair in tag_transition_dictionary:
                tag_transition_dictionary[pair] += 1
            else:
                tag_transition_dictionary[pair] = 1

    # print(tag_transition_dictionary)
    return tag_transition_dictionary

In [20]:
def Create_transition_matrix(tag_transition_dictionary, tag_list, tag_count, alpha = 1):
    
    transition_matrix = np.zeros((len(tag_list), len(tag_list)))
    for i in range(len(tag_list)):       
        for j in range(len(tag_list)):
            pair = (tag_list[i],tag_list[j])
            if pair in tag_transition_dictionary:
                transition_matrix[i][j] = (tag_transition_dictionary[pair] + alpha)/(tag_count[pair[0]] + alpha * len(tag_list))            
            else:
                if(pair[0] == "EOS"):
                    transition_matrix[i][j] = 1/len(tag_list)
                    continue                
                transition_matrix[i][j] = (alpha)/(tag_count[pair[0]] + alpha * len(tag_list))    
    df = pd.DataFrame(transition_matrix, index = tag_list, columns = tag_list)
    return transition_matrix


In [21]:
def Create_emission_matrix(word_tag_dictionary, tag_list, word_list, tag_count):   
    alpha = 1
    emission_matrix = np.zeros((len(tag_list), len(word_list)))
    for i in range(len(tag_list)):        
        for j in range(len(word_list)):
            pair = (word_list[j],tag_list[i])
            if pair in word_tag_dictionary:
                emission_matrix[i][j] = (word_tag_dictionary[pair] + alpha)/(tag_count[pair[1]] + alpha * len(word_list))           
            else:
                emission_matrix[i][j] = (alpha)/(tag_count[pair[1]] + alpha * len(word_list))
    
    df = pd.DataFrame(emission_matrix, index = tag_list, columns = word_list)
    # print(df)

    return emission_matrix


In [26]:
def viterbi_algorithm(transition_matrix, emission_matrix, observation_sequence, tag_vocabulary, word_vocabulary):
    num_tags = len(tag_vocabulary)
    num_observations = len(observation_sequence)
    
    # Initialize Viterbi and backpointer matrices
    viterbi_scores = np.zeros((num_tags, num_observations))
    backpointers = np.zeros((num_tags, num_observations), dtype=int)

    # Initialization step
    for tag_index in range(num_tags):
        try:
            word_index = word_vocabulary.index(observation_sequence[0][0])
        except ValueError:
            word_index = word_vocabulary.index("OOV")  

        viterbi_scores[tag_index, 0] = transition_matrix[tag_vocabulary.index("BOS"), tag_index] * emission_matrix[tag_index, word_index]
        backpointers[tag_index, 0] = tag_vocabulary.index("BOS")

    # Recursion step
    for observation_index in range(1, num_observations):
        for tag_index in range(num_tags):
            try:
                word_index = word_vocabulary.index(observation_sequence[observation_index][0])
            except ValueError:
                word_index = word_vocabulary.index("OOV") 

            scores = [viterbi_scores[prev_tag_index, observation_index - 1] * transition_matrix[prev_tag_index, tag_index] * emission_matrix[tag_index, word_index] for prev_tag_index in range(num_tags)]
            viterbi_scores[tag_index, observation_index] = max(scores)
            backpointers[tag_index, observation_index] = np.argmax(scores)

    # Termination step
    final_scores = [viterbi_scores[tag_index, num_observations - 1] * transition_matrix[tag_index, tag_vocabulary.index("EOS")] for tag_index in range(num_tags)]
    best_final_score = max(final_scores)
    best_final_tag_index = np.argmax(final_scores)
    best_path_indices = [best_final_tag_index]

    # Backtracking step
    for observation_index in range(num_observations - 2, -1, -1):
        best_final_tag_index = backpointers[best_final_tag_index, observation_index + 1]
        best_path_indices.insert(0, best_final_tag_index)  # Insert at the beginning

    best_path = ["BOS"] + [tag_vocabulary[tag_index] for tag_index in best_path_indices] + ["EOS"]

    return best_final_score, best_path  # Return the best score and path

In [23]:
def test_algo_accuracy(data_test, transition_matrix, emision_matrix, tag_list, word_list):
    correct_predictions_sentence = 0  
    correct_predictions_tags = 0  
    total_test_tags = 0 

    for test_x in data_test:
        max_prob, path = viterbi_algorithm(transition_matrix, emision_matrix, test_x, tag_list, word_list)
        expected_tags = [test_x[i][1] for i in range(len(test_x))]  # Extract correct tags

   
        # print(f"Test sentence: {[word[0] for word in test_x]}")  # Print only words
        # print(f"Expected tags: {expected_tags}")
        # print(f"Predicted tags: {path}")
        # print(f"Probability: {max_prob}")

        if expected_tags == path:
            correct_predictions_sentence += 1

        correct_predictions_tags += sum(1 for a, b in zip(expected_tags, path) if a == b)
        total_test_tags += len(expected_tags)

    sentence_accuracy = (correct_predictions_sentence / len(data_test)) * 100
    tag_accuracy = (correct_predictions_tags / total_test_tags) * 100

    return sentence_accuracy, tag_accuracy


In [27]:
expected_tag_indices = []
predicted_tag_indices = []

def classification_list(test_data, transition_probs, emission_probs, tag_set, vocab):
    for sentence in test_data:
        # Get predicted tags using Viterbi algorithm
        _, predicted_tags = viterbi_algorithm(transition_probs, emission_probs, sentence, tag_set, vocab)
        
        # Get true tags from the sentence
        true_tags = [word_tag[1] for word_tag in sentence]
        
        # Convert tags to indices
        expected_indices = [tag_set.index(tag) for tag in true_tags]
        predicted_indices = [tag_set.index(tag) for tag in predicted_tags]
        
        # AppEOS to global lists
        expected_tag_indices.appEOS(expected_indices)
        predicted_tag_indices.appEOS(predicted_indices)
    


In [ ]:
folds = []
k = 5
fold_size = len(tagged_sentences) // k
for a in range(5) :
    random.shuffle(tagged_sentences)
    folds.appEOS([tagged_sentences[i * fold_size : (i + 1) * fold_size] for i in range(5)])

data_train = []
data_test = []

for i in range(5):
    data_train.appEOS(folds[i][0] + folds[i][1] + folds[i][2] + folds[i][3])
    data_test.appEOS(folds[i][4])

   
    word_tag_dictionary, word_list, tag_count, tag_list = useful_dictionary_or_list(data_train[i])
    tag_transition_dictionary = create_tag_transition_dictionary(data_train[i])
    transition_matrix = Create_transition_matrix(tag_transition_dictionary, tag_list, tag_count, alpha = 1)
    emision_matrix = Create_emission_matrix(word_tag_dictionary, tag_list, word_list, tag_count)

    #check matrix implementation
    print(np.sum(emision_matrix, axis = 1))
    print(np.sum(transition_matrix, axis = 1))

    classification_list(data_test[i], transition_matrix, emision_matrix, tag_list, word_list)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Accuracy of the model on fold 1: 35.94349494244855, 93.90912598354618
---------------------------------------------

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Accuracy of the model on fold 2: 35.72549703522846, 93.87068049740674
---------------------------------------------



In [47]:
temp_list_actual = []
temp_list_expected = []
for row in classification_list_actual:
    temp_list_actual += row

for row in classification_list_expected:
    temp_list_expected += row

print(classification_report(temp_list_actual, temp_list_expected, target_names=tag_list))

['BOS', 'PRON', 'VERB', 'DET', 'NOUN', 'CONJ', 'PRT', 'ADV', 'ADP', '.', 'EOS', 'ADJ', 'NUM', 'X']
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         5
           2       0.83      0.91      0.87        11
           3       1.00      1.00      1.00         6
           4       0.87      0.93      0.90        14
           5       1.00      1.00      1.00         5
           6       1.00      1.00      1.00         6
           7       0.83      0.83      0.83         6
           8       1.00      0.80      0.89         5
           9       1.00      1.00      1.00        17
          10       1.00      0.71      0.83         7
          11       1.00      1.00      1.00         3
          12       0.67      1.00      0.80         2

    accuracy                           0.93        92
   macro avg       0.94      0.94      0.93        92
weighted avg       0.94      0.93  

### VITERBI ALGO ON WHOLE DATABASE

In [11]:
word_tag_dictionary, word_list, tag_count, tag_list = useful_dictionary_or_list(tagged_sentences)
tag_transition_dictionary = create_tag_transition_dictionary(tagged_sentences)
transition_matrix = Create_transition_matrix(tag_transition_dictionary, tag_list, tag_count, alpha = 1)
emision_matrix = Create_emission_matrix(word_tag_dictionary, tag_list, word_list, tag_count)

#check matrix implementation
print(np.sum(emision_matrix, axis = 1))
print(np.sum(transition_matrix, axis = 1))

#test data with accracy #### Accuracy of the model on 80% of whole data: 33.214161144053016, 93.55473283734179
accuracy_sentence, accuracy_tags = test_algo_accuracy(tagged_sentences[:5], transition_matrix, emision_matrix, tag_list, word_list)
print(f"Accuracy of the model on whole data: {accuracy_sentence}, {accuracy_tags}")
print("---------------------------------------------")
print()

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Test sentence: ['BOS', 'I', 'stopped', 'the', 'tractor', 'and', 'climbed', 'down', 'and', 'the', 'boy', 'came', 'eagerly', 'up', 'to', 'me', '.', 'EOS']
Expected tags: ['BOS', 'PRON', 'VERB', 'DET', 'NOUN', 'CONJ', 'VERB', 'PRT', 'CONJ', 'DET', 'NOUN', 'VERB', 'ADV', 'PRT', 'ADP', 'PRON', '.', 'EOS']
Predicted tags: ['BOS', 'PRON', 'VERB', 'DET', 'NOUN', 'CONJ', 'VERB', 'PRT', 'CONJ', 'DET', 'NOUN', 'VERB', 'ADV', 'PRT', 'ADP', 'PRON', '.', 'EOS']
Probability: 2.0848381241804144e-47
Test sentence: ['BOS', 'His', 'objective', 'is', 'merely', 'to', 'determine', '``', 'what', 'distinctions', 'of', 'length', 'and', 'syllabicity', 'it', 'may', 'be', 'desirable', 'to', 'make', 'explicit', 'in', 'a', 'Kikuyu', 'orthography', "''", '(', '59', ')', '.', 'EOS']
Expected tags: ['BOS', 'DET', 'NOUN', 'VERB', 'ADV', 'PRT', 'VERB', '.', 'DET', 'NOUN', 'ADP', 'NOUN', 'CONJ', 'NOUN', 'PRON', 'VERB', 'VERB', 'ADJ', 